# Week 2 Lab: Finance Theory in Practice

**BUS 696: Generative AI in Finance**  
**Professor Jonathan Hersh**

---

## Learning Objectives

By the end of this lab, you will be able to:
1. Test whether stock returns are normally distributed using histograms, QQ plots, and statistical tests
2. Calculate a stock's **beta** and **alpha** using CAPM regression
3. Plot option payoff diagrams for calls and puts
4. Explore real options chain data from Yahoo Finance
5. Build a simple two-stock portfolio and visualize the diversification benefit
6. Write effective prompts to interpret financial analysis results

---

## Part 1: Setup

In [ ]:
# Install if needed (run once, then comment out)
# !pip install yfinance pandas matplotlib plotly scipy statsmodels

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries loaded!")

In [ ]:
# Download data we'll use throughout the lab
# Apple (our stock) and S&P 500 (the market benchmark)
apple = yf.download('AAPL', period='2y')
apple.columns = apple.columns.get_level_values(0)

spy = yf.download('SPY', period='2y')   # SPY = S&P 500 ETF
spy.columns = spy.columns.get_level_values(0)

# Calculate daily returns
apple['Return'] = apple['Close'].pct_change()
spy['Return'] = spy['Close'].pct_change()

# Calculate log returns
apple['Log_Return'] = np.log(apple['Close'] / apple['Close'].shift(1))
spy['Log_Return'] = np.log(spy['Close'] / spy['Close'].shift(1))

print(f"Apple: {len(apple)} trading days from {apple.index[0].date()} to {apple.index[-1].date()}")
print(f"SPY:   {len(spy)} trading days from {spy.index[0].date()} to {spy.index[-1].date()}")

---

## Part 2: Are Stock Returns Normally Distributed?

Many financial models assume returns follow a **normal (Gaussian) distribution**. Let's test that assumption with real data.

If returns were truly normal:
- 68% would fall within 1 standard deviation of the mean
- 95% within 2 standard deviations
- 99.7% within 3 standard deviations
- A 4+ sigma event would be incredibly rare

### Histogram vs. Normal Distribution

In [ ]:
returns = apple['Return'].dropna()
mu = returns.mean()
sigma = returns.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Apple returns
axes[0].hist(returns, bins=60, density=True, color='steelblue', edgecolor='white', alpha=0.7, label='Actual returns')
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal distribution')
axes[0].set_title('Apple (AAPL) Daily Returns vs. Normal Distribution', fontsize=12)
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: S&P 500 returns
spy_returns = spy['Return'].dropna()
mu_spy = spy_returns.mean()
sigma_spy = spy_returns.std()
axes[1].hist(spy_returns, bins=60, density=True, color='darkgreen', edgecolor='white', alpha=0.7, label='Actual returns')
x2 = np.linspace(mu_spy - 4*sigma_spy, mu_spy + 4*sigma_spy, 200)
axes[1].plot(x2, stats.norm.pdf(x2, mu_spy, sigma_spy), 'r-', linewidth=2, label='Normal distribution')
axes[1].set_title('S&P 500 (SPY) Daily Returns vs. Normal Distribution', fontsize=12)
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

**Notice:** The actual returns have:
- A **taller, narrower peak** than the normal curve (more "average" days)
- **Fatter tails** (more extreme days than the normal curve predicts)

This is called a **leptokurtic** distribution.

### QQ Plot

A **QQ (Quantile-Quantile) plot** compares the actual return distribution against what we'd expect if returns were perfectly normal. If returns were normal, all points would fall on the red diagonal line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Apple QQ plot
stats.probplot(returns, dist='norm', plot=axes[0])
axes[0].set_title('QQ Plot: Apple (AAPL) Returns', fontsize=12)
axes[0].get_lines()[0].set_color('steelblue')
axes[0].get_lines()[0].set_markersize(3)

# SPY QQ plot
stats.probplot(spy_returns, dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot: S&P 500 (SPY) Returns', fontsize=12)
axes[1].get_lines()[0].set_color('darkgreen')
axes[1].get_lines()[0].set_markersize(3)

plt.tight_layout()
plt.show()

**Interpretation:** If the dots follow the red line, returns are normally distributed. Where the dots curve *away* from the line at the edges, that's the **fat tails** — extreme returns that are more common than a normal distribution would predict.

### Statistical Tests for Normality

In [ ]:
# Jarque-Bera test: tests whether skewness and kurtosis match a normal distribution
# H0 (null hypothesis): the data is normally distributed
# If p-value < 0.05, we reject H0 — the data is NOT normal

for name, ret in [('Apple (AAPL)', returns), ('S&P 500 (SPY)', spy_returns)]:
    skew = ret.skew()
    kurt = ret.kurtosis()  # excess kurtosis (normal = 0)
    jb_stat, jb_pval = stats.jarque_bera(ret)
    
    print(f"\n{name}")
    print(f"  Skewness:         {skew:.3f}  (normal = 0)")
    print(f"  Excess Kurtosis:  {kurt:.3f}  (normal = 0, fat tails > 0)")
    print(f"  Jarque-Bera stat: {jb_stat:.2f}")
    print(f"  p-value:          {jb_pval:.6f}")
    if jb_pval < 0.05:
        print(f"  Conclusion: REJECT normality (p < 0.05) — returns are NOT normally distributed")
    else:
        print(f"  Conclusion: Cannot reject normality (p >= 0.05)")

### Counting Extreme Events

In [ ]:
# How many extreme days actually occurred vs. what a normal distribution predicts?
n_days = len(returns)

print("Extreme Events: Actual vs. Normal Distribution Prediction")
print("=" * 60)
print(f"Total trading days: {n_days}\n")

for n_sigma in [2, 3, 4]:
    # Actual count of days beyond n sigma
    actual = ((returns > mu + n_sigma * sigma) | (returns < mu - n_sigma * sigma)).sum()
    # Expected under normal distribution
    expected_pct = 2 * (1 - stats.norm.cdf(n_sigma))
    expected = expected_pct * n_days
    
    print(f"Beyond {n_sigma} sigma:")
    print(f"  Normal predicts: {expected:.1f} days")
    print(f"  Actually saw:    {actual} days")
    if expected > 0:
        print(f"  Ratio:           {actual/expected:.1f}x more than expected")
    print()

**Takeaway:** Extreme events happen *far more often* than normal distribution models predict. This is why risk management based purely on normal assumptions can be dangerously wrong.

---

## Part 3: CAPM — Calculating Beta and Alpha

The **Capital Asset Pricing Model** says:

$$E(R_i) = R_f + \beta_i \times (E(R_m) - R_f)$$

To calculate beta empirically, we run a **regression** of stock returns on market returns:

$$R_{\text{stock}} = \alpha + \beta \times R_{\text{market}} + \epsilon$$

- **Beta (slope)** = how much the stock moves when the market moves
- **Alpha (intercept)** = excess return beyond what CAPM predicts

In [ ]:
# Align the data: make sure we have matching dates
merged = pd.DataFrame({
    'AAPL': apple['Return'],
    'SPY': spy['Return']
}).dropna()

print(f"Matched trading days: {len(merged)}")
merged.tail()

In [ ]:
# Run the CAPM regression
slope, intercept, r_value, p_value, std_err = stats.linregress(merged['SPY'], merged['AAPL'])

beta = slope
alpha_daily = intercept
alpha_annual = alpha_daily * 252  # annualize
r_squared = r_value ** 2

print("CAPM Regression: Apple vs. S&P 500")
print("=" * 45)
print(f"Beta:              {beta:.3f}")
print(f"Alpha (daily):     {alpha_daily:.6f} ({alpha_daily*100:.4f}%)")
print(f"Alpha (annualized): {alpha_annual:.4f} ({alpha_annual*100:.2f}%)")
print(f"R-squared:         {r_squared:.3f}")
print(f"p-value (beta):    {p_value:.2e}")
print()

if beta > 1:
    print(f"Beta = {beta:.2f} > 1: Apple is MORE volatile than the market.")
    print(f"When the market moves 1%, Apple tends to move {beta:.2f}%.")
elif beta < 1:
    print(f"Beta = {beta:.2f} < 1: Apple is LESS volatile than the market.")
    print(f"When the market moves 1%, Apple tends to move {beta:.2f}%.")
else:
    print(f"Beta ≈ 1: Apple moves roughly in line with the market.")

print(f"\nR-squared = {r_squared:.3f}: The market explains {r_squared*100:.1f}% of Apple's return variation.")
print(f"The remaining {(1-r_squared)*100:.1f}% is company-specific (idiosyncratic) risk.")

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(8, 8))
plt.scatter(merged['SPY'], merged['AAPL'], alpha=0.3, s=12, color='steelblue', label='Daily returns')

# Regression line
x_line = np.linspace(merged['SPY'].min(), merged['SPY'].max(), 100)
y_line = intercept + beta * x_line
plt.plot(x_line, y_line, 'r-', linewidth=2, label=f'CAPM: α={alpha_daily:.4f}, β={beta:.2f}')

plt.axhline(y=0, color='gray', linewidth=0.5)
plt.axvline(x=0, color='gray', linewidth=0.5)
plt.xlabel('S&P 500 (SPY) Daily Return', fontsize=12)
plt.ylabel('Apple (AAPL) Daily Return', fontsize=12)
plt.title(f'CAPM Regression: Apple vs. Market (β = {beta:.2f}, R² = {r_squared:.3f})', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Security Market Line

Let's calculate beta for several stocks and plot the **Security Market Line** (SML) — the relationship between beta and expected return.

In [ ]:
# Calculate beta and return for several stocks
tickers = ['AAPL', 'MSFT', 'TSLA', 'JNJ', 'XOM', 'NVDA', 'JPM', 'NEE']
stock_data = yf.download(tickers, period='2y')['Close']

# SPY returns as market benchmark
spy_ret = spy['Return'].dropna()

results = []
for ticker in tickers:
    stock_ret = stock_data[ticker].pct_change().dropna()
    # Align dates
    common = pd.DataFrame({'stock': stock_ret, 'market': spy_ret}).dropna()
    slope_i, intercept_i, _, _, _ = stats.linregress(common['market'], common['stock'])
    ann_return = common['stock'].mean() * 252
    results.append({'Ticker': ticker, 'Beta': slope_i, 'Ann. Return': ann_return})

sml_df = pd.DataFrame(results)
print(sml_df.to_string(index=False))

In [ ]:
# Plot the Security Market Line
risk_free = 0.05
market_return = spy_ret.mean() * 252

plt.figure(figsize=(10, 7))

# SML line: from risk-free rate through the market portfolio (beta=1)
beta_range = np.linspace(0, 2.5, 100)
sml_line = risk_free + beta_range * (market_return - risk_free)
plt.plot(beta_range, sml_line, 'r--', linewidth=2, label='Security Market Line (SML)', alpha=0.7)

# Plot each stock
for _, row in sml_df.iterrows():
    color = 'green' if row['Ann. Return'] > risk_free + row['Beta'] * (market_return - risk_free) else 'red'
    plt.scatter(row['Beta'], row['Ann. Return'], s=100, color=color, zorder=5, edgecolor='black')
    plt.annotate(row['Ticker'], (row['Beta'], row['Ann. Return']),
                 textcoords='offset points', xytext=(8, 5), fontsize=11, fontweight='bold')

# Market portfolio
plt.scatter(1.0, market_return, s=150, color='gold', zorder=5, edgecolor='black', marker='*')
plt.annotate('Market (SPY)', (1.0, market_return), textcoords='offset points', xytext=(8, 5), fontsize=10)

plt.xlabel('Beta', fontsize=12)
plt.ylabel('Annualized Return', fontsize=12)
plt.title('Security Market Line', fontsize=14)
plt.legend(fontsize=11)
plt.axhline(y=risk_free, color='gray', linewidth=0.5, linestyle=':')
plt.text(0.05, risk_free + 0.005, f'Risk-free = {risk_free:.0%}', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

print("Green dots = ABOVE the SML (positive alpha — outperforming for their risk level)")
print("Red dots   = BELOW the SML (negative alpha — underperforming for their risk level)")

---

## Part 4: Options — Payoff Diagrams

An **option** gives you the right (not obligation) to buy or sell a stock at a specific price.

- **Call option**: Right to **buy** at the strike price
- **Put option**: Right to **sell** at the strike price

Let's visualize the payoff diagrams.

In [ ]:
# Option payoff diagrams
strike = 180       # Strike price
call_premium = 5   # Cost of buying the call
put_premium = 4    # Cost of buying the put

# Range of possible stock prices at expiration
prices = np.linspace(150, 210, 200)

# Call payoff: max(Stock - Strike, 0) - Premium
call_payoff = np.maximum(prices - strike, 0) - call_premium

# Put payoff: max(Strike - Stock, 0) - Premium
put_payoff = np.maximum(strike - prices, 0) - put_premium

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Call option
axes[0].plot(prices, call_payoff, 'b-', linewidth=2)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=strike, color='gray', linewidth=0.5, linestyle='--')
axes[0].fill_between(prices, call_payoff, 0, where=(call_payoff > 0), alpha=0.2, color='green')
axes[0].fill_between(prices, call_payoff, 0, where=(call_payoff < 0), alpha=0.2, color='red')
axes[0].set_title(f'Call Option Payoff (Strike=${strike}, Premium=${call_premium})', fontsize=12)
axes[0].set_xlabel('Stock Price at Expiration ($)')
axes[0].set_ylabel('Profit / Loss ($)')
axes[0].annotate(f'Break-even: ${strike + call_premium}', xy=(strike + call_premium, 0),
                 xytext=(strike + call_premium + 5, 5), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='black'))
axes[0].annotate(f'Max loss: -${call_premium}', xy=(160, -call_premium),
                 fontsize=10, color='red')

# Put option
axes[1].plot(prices, put_payoff, 'r-', linewidth=2)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=strike, color='gray', linewidth=0.5, linestyle='--')
axes[1].fill_between(prices, put_payoff, 0, where=(put_payoff > 0), alpha=0.2, color='green')
axes[1].fill_between(prices, put_payoff, 0, where=(put_payoff < 0), alpha=0.2, color='red')
axes[1].set_title(f'Put Option Payoff (Strike=${strike}, Premium=${put_premium})', fontsize=12)
axes[1].set_xlabel('Stock Price at Expiration ($)')
axes[1].set_ylabel('Profit / Loss ($)')
axes[1].annotate(f'Break-even: ${strike - put_premium}', xy=(strike - put_premium, 0),
                 xytext=(strike - put_premium - 15, 5), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='black'))
axes[1].annotate(f'Max loss: -${put_premium}', xy=(195, -put_premium),
                 fontsize=10, color='red')

plt.tight_layout()
plt.show()

**Key observations:**
- **Call**: You profit when the stock goes *above* the strike + premium. Max loss = the premium.
- **Put**: You profit when the stock goes *below* the strike - premium. Max loss = the premium.
- Both have the characteristic **hockey stick** shape — asymmetric risk/reward.
- This asymmetry is why options are powerful for hedging and speculation.

### Combined Strategies: Protective Put (Insurance)

What if you own Apple stock and buy a put option to protect against a crash? This is called a **protective put** — it's like buying insurance on your stock.

In [ ]:
# You own the stock (bought at $180) AND you buy a put at strike $180
stock_payoff = prices - 180  # P&L from owning the stock
protective_put = stock_payoff + put_payoff  # stock + put combined

plt.figure(figsize=(10, 6))
plt.plot(prices, stock_payoff, '--', color='steelblue', linewidth=1.5, label='Stock only', alpha=0.7)
plt.plot(prices, put_payoff, '--', color='red', linewidth=1.5, label='Put only', alpha=0.7)
plt.plot(prices, protective_put, 'k-', linewidth=2.5, label='Protective put (stock + put)')
plt.axhline(y=0, color='gray', linewidth=0.5)
plt.axvline(x=strike, color='gray', linewidth=0.5, linestyle=':')
plt.fill_between(prices, protective_put, 0, where=(protective_put > 0), alpha=0.15, color='green')
plt.fill_between(prices, protective_put, 0, where=(protective_put < 0), alpha=0.15, color='red')
plt.xlabel('Stock Price at Expiration ($)', fontsize=12)
plt.ylabel('Profit / Loss ($)', fontsize=12)
plt.title('Protective Put: Stock + Put Option = Downside Insurance', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"With a protective put, your max loss is limited to ${put_premium} (the put premium).")
print(f"But you keep all the upside above ${strike + put_premium}.")
print(f"The cost: you give up ${put_premium} of profit in the good scenario.")

### Exploring Real Options Data

Yahoo Finance provides real options chain data. Let's look at what options are actually available for Apple.

In [ ]:
# Get Apple's available options expiration dates
aapl = yf.Ticker('AAPL')
expirations = aapl.options
print("Available expiration dates:")
for i, exp in enumerate(expirations[:8]):  # show first 8
    print(f"  {i+1}. {exp}")
print(f"  ... and {len(expirations) - 8} more")

In [ ]:
# Get the options chain for the nearest expiration
nearest_exp = expirations[0]
opt_chain = aapl.option_chain(nearest_exp)

print(f"Options chain for AAPL expiring {nearest_exp}")
print(f"\nCALL options (right to BUY):")
# Show calls near the current stock price
current_price = apple['Close'].iloc[-1]
calls = opt_chain.calls
near_money_calls = calls[(calls['strike'] >= current_price * 0.95) & 
                         (calls['strike'] <= current_price * 1.05)]
print(near_money_calls[['strike', 'lastPrice', 'bid', 'ask', 'volume', 'impliedVolatility']].to_string(index=False))

print(f"\nPUT options (right to SELL):")
puts = opt_chain.puts
near_money_puts = puts[(puts['strike'] >= current_price * 0.95) & 
                       (puts['strike'] <= current_price * 1.05)]
print(near_money_puts[['strike', 'lastPrice', 'bid', 'ask', 'volume', 'impliedVolatility']].to_string(index=False))

**Reading the options chain:**
- **strike**: The price at which you can buy/sell the stock
- **lastPrice**: What the option last traded at
- **bid/ask**: Current buy and sell prices (just like stocks!)
- **volume**: How many contracts traded today
- **impliedVolatility**: The market's expectation of future volatility, *derived from the option price*

Notice that **implied volatility** is embedded in option prices — this is one of the most important signals in finance. AI models can analyze implied volatility patterns to forecast future market movements.

---

## Part 5: Portfolio Diversification

What happens when you combine two stocks in a portfolio? If they're not perfectly correlated, the portfolio risk is **less** than the average of the two stocks' risks.

This is diversification — the "free lunch" of finance.

In [ ]:
# Download two stocks
pair = yf.download(['AAPL', 'XOM'], period='2y')['Close']
pair_returns = pair.pct_change().dropna()

corr = pair_returns['AAPL'].corr(pair_returns['XOM'])
print(f"Correlation between Apple and ExxonMobil: {corr:.3f}")
print("(Lower correlation = more diversification benefit)\n")

# Individual stock stats (annualized)
for ticker in ['AAPL', 'XOM']:
    ret = pair_returns[ticker].mean() * 252
    vol = pair_returns[ticker].std() * np.sqrt(252)
    print(f"{ticker}: Return = {ret:.2%}, Volatility = {vol:.2%}")

In [ ]:
# Vary the portfolio weights and compute risk-return for each
weights_aapl = np.linspace(0, 1, 100)

mu_aapl = pair_returns['AAPL'].mean() * 252
mu_xom = pair_returns['XOM'].mean() * 252
vol_aapl = pair_returns['AAPL'].std() * np.sqrt(252)
vol_xom = pair_returns['XOM'].std() * np.sqrt(252)

port_returns = []
port_vols = []

for w in weights_aapl:
    # Portfolio return = weighted average of individual returns
    p_ret = w * mu_aapl + (1 - w) * mu_xom
    
    # Portfolio volatility — THIS is where diversification happens
    # It's NOT just the weighted average of volatilities
    p_vol = np.sqrt(
        (w * vol_aapl)**2 + 
        ((1 - w) * vol_xom)**2 + 
        2 * w * (1 - w) * vol_aapl * vol_xom * corr
    )
    
    port_returns.append(p_ret)
    port_vols.append(p_vol)

# Find the minimum volatility portfolio
min_vol_idx = np.argmin(port_vols)
min_vol_w = weights_aapl[min_vol_idx]

plt.figure(figsize=(10, 7))
plt.plot(port_vols, port_returns, 'b-', linewidth=2, label='Portfolio combinations')

# Mark individual stocks
plt.scatter(vol_aapl, mu_aapl, s=150, color='steelblue', zorder=5, edgecolor='black')
plt.annotate('100% AAPL', (vol_aapl, mu_aapl), textcoords='offset points', xytext=(10, 5), fontsize=11, fontweight='bold')

plt.scatter(vol_xom, mu_xom, s=150, color='green', zorder=5, edgecolor='black')
plt.annotate('100% XOM', (vol_xom, mu_xom), textcoords='offset points', xytext=(10, 5), fontsize=11, fontweight='bold')

# Mark minimum volatility portfolio
plt.scatter(port_vols[min_vol_idx], port_returns[min_vol_idx], s=200, color='gold', zorder=5, 
            edgecolor='black', marker='*')
plt.annotate(f'Min Risk\n({min_vol_w:.0%} AAPL, {1-min_vol_w:.0%} XOM)', 
             (port_vols[min_vol_idx], port_returns[min_vol_idx]),
             textcoords='offset points', xytext=(-80, -25), fontsize=10)

plt.xlabel('Annualized Volatility (Risk)', fontsize=12)
plt.ylabel('Annualized Return', fontsize=12)
plt.title('Portfolio Frontier: Apple + ExxonMobil', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Minimum risk portfolio: {min_vol_w:.0%} Apple + {1-min_vol_w:.0%} ExxonMobil")
print(f"  Return:     {port_returns[min_vol_idx]:.2%}")
print(f"  Volatility: {port_vols[min_vol_idx]:.2%}")
print(f"\nNotice: the minimum risk portfolio has LESS volatility than either stock alone!")
print(f"This is the diversification benefit.")

**Key insight:** The curved line shows all possible combinations of the two stocks. The curve bends to the *left* of a straight line between the two stocks — meaning you can get *less risk* than a simple average by combining them. The lower the correlation, the more the curve bends left (more diversification benefit).

This is a preview of **Mean-Variance Portfolio Theory** (Markowitz). With many stocks, you'd get an **efficient frontier** — the boundary of the best possible risk-return combinations.

---

## Part 6: Your Turn — Exercises

### Exercise 1: CAPM for a Stock of Your Choice

Pick a stock and calculate its beta and alpha using CAPM regression against SPY.

Some ideas: `TSLA` (high beta?), `JNJ` (low beta?), `NVDA` (tech growth), `GS` (financial sector)

In [ ]:
# YOUR CODE HERE
# 1. Download your chosen stock (period='2y')
# 2. Flatten MultiIndex columns
# 3. Calculate daily returns
# 4. Merge with spy returns
# 5. Run linregress to get beta and alpha
# 6. Print results and interpret



### Exercise 2: Option Payoff for a Straddle

A **straddle** is when you buy BOTH a call AND a put at the same strike price. You profit when the stock makes a big move in *either* direction.

Plot the payoff diagram for a straddle with strike = $180, call premium = $5, put premium = $4.

In [ ]:
# YOUR CODE HERE
# Hint: straddle_payoff = call_payoff + put_payoff
# Total premium = call_premium + put_premium
# You profit when the stock moves more than the total premium in either direction



### Exercise 3: Test Normality for a Different Stock

Pick a different stock (or an ETF like `QQQ`, `IWM`, `GLD`) and test whether its returns are normally distributed. Create the histogram, QQ plot, and run the Jarque-Bera test.

In [ ]:
# YOUR CODE HERE



---

## Part 7: Prompt Engineering

Now use AI to help interpret what you've learned.

### Exercise 4: Interpret Your CAPM Results

Take the beta and alpha values you calculated (either for Apple or your chosen stock) and write a prompt asking an AI to explain what they mean in plain English.

**Include the actual numbers.** A good prompt might also ask: "What would this mean for a portfolio manager?" or "How does this compare to the average stock?"

```
YOUR PROMPT HERE:



```

**AI Response:** (Paste response here)

```



```

**Your evaluation:** Was the response accurate given what you learned today?